# U.S. Analytics Job Market — API Exploration

## Objective

This notebook explores the Adzuna Jobs API and establishes the initial data ingestion workflow for the U.S. Analytics Job Market Intelligence project.

The goal is to:

- Connect to the Adzuna API securely
- Request current U.S. Data Analyst job postings
- Inspect the API response structure
- Convert job records into a Pandas DataFrame

## 1. Environment Setup

Import the Python libraries required for API requests, credential management, and data analysis.

In [8]:
import requests
import os
import pandas as pd

from dotenv import load_dotenv


## 2. Load API Credentials

API credentials are stored in a local `.env` file and excluded from version control to avoid exposing sensitive information.

In [14]:
load_dotenv("../.env")

app_id = os.getenv("ADZUNA_APP_ID")
app_key = os.getenv("ADZUNA_APP_KEY")

print(app_id is not None)
print(app_key is not None)

True
True


## 3. Configure the API Request

The initial request retrieves the first page of current U.S. Data Analyst job postings. A small sample is used first to understand the API response before scaling the ingestion process.

In [15]:
url = "https://api.adzuna.com/v1/api/jobs/us/search/1"


In [16]:
params = {
    "app_id": app_id,
    "app_key": app_key,
    "results_per_page": 10,
    "what": "data analyst",
}


## 4. Send the API Request

Send a GET request to the Adzuna Jobs API and verify that the server responds successfully.

In [17]:
response = requests.get(url, params=params)


In [18]:
response.status_code


200

## 5. Inspect the JSON Response

The API returns job data in JSON format. Before converting the records into a DataFrame, inspect the structure of the response to understand how the data is organized.

In [19]:
data = response.json()


In [20]:
type(data)


dict

In [21]:
data.keys()


dict_keys(['results', '__CLASS__', 'count', 'mean'])

In [22]:
type(data["results"])


list

In [23]:
len(data["results"])


10

In [24]:
first_job = data["results"][0]


In [25]:
first_job.keys()


dict_keys(['company', 'category', 'created', 'id', 'salary_is_predicted', '__CLASS__', 'salary_min', 'title', 'salary_max', 'description', 'longitude', 'adref', 'latitude', 'location', 'redirect_url'])

### Response Structure

The top-level API response is a Python dictionary. The `results` field contains a list of individual job postings, with each posting represented as its own dictionary.

Structure:

`API Response → Dictionary → results → List → Individual Job`

In [26]:
print(first_job["title"])


Data Analyst


In [28]:
first_job["company"]["display_name"]


'Altair Data Resources, Inc.'

In [29]:
{
    "title": first_job.get("title"),
    "company": first_job.get("company", {}).get("display_name"),
    "location": first_job.get("location", {}).get("display_name"),
    "created": first_job.get("created"),
    "salary_min": first_job.get("salary_min"),
    "salary_max": first_job.get("salary_max"),
}


{'title': 'Data Analyst',
 'company': 'Altair Data Resources, Inc.',
 'location': 'Brentwood, Williamson County',
 'created': '2026-09-25T21:59:05Z',
 'salary_min': 60000,
 'salary_max': 80000}

## 6. Convert Job Records to a DataFrame

The API returns nested JSON records. `pandas.json_normalize()` is used to flatten nested fields such as company and location into a tabular structure suitable for analysis.

In [30]:
jobs = data["results"]

df = pd.json_normalize(jobs)


In [31]:
df.shape


(10, 21)

In [32]:
df.columns.tolist()


['created',
 'id',
 'salary_is_predicted',
 '__CLASS__',
 'salary_min',
 'title',
 'salary_max',
 'description',
 'longitude',
 'adref',
 'latitude',
 'redirect_url',
 'company.__CLASS__',
 'company.display_name',
 'category.__CLASS__',
 'category.tag',
 'category.label',
 'location.area',
 'location.display_name',
 'location.__CLASS__',
 'contract_time']

In [33]:
df.head()


,created,id,salary_is_predicted,__CLASS__,salary_min,title,salary_max,description,longitude,adref,...,redirect_url,company.__CLASS__,company.display_name,category.__CLASS__,category.tag,category.label,location.area,location.display_name,location.__CLASS__,contract_time
0,2026-09-25T21:59:05Z,5899042199,0,Adzuna::API::Response::Job,60000.00,Data Analyst,80000.00,Description: Altair Data Resources is seeking ...,-86.774532,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTg5OTA0MjE5OSIsI...,...,https://www.adzuna.com/details/5899042199?utm_...,Adzuna::API::Response::Company,"Altair Data Resources, Inc.",Adzuna::API::Response::Category,it-jobs,IT Jobs,"[US, Tennessee, Williamson County, Brentwood]","Brentwood, Williamson County",Adzuna::API::Response::Location,NaN
1,2026-08-24T02:59:05Z,5854221865,1,Adzuna::API::Response::Job,76953.42,Privacy Program Data Analyst (HIPAA),76953.42,Job Summary : The Privacy Program Data Analyst...,-73.775133,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTg1NDIyMTg2NSIsI...,...,https://www.adzuna.com/land/ad/5854221865?se=N...,Adzuna::API::Response::Company,WMCHealth,Adzuna::API::Response::Category,healthcare-nursing-jobs,Healthcare & Nursing Jobs,"[US, New York, Westchester County, Valhalla]","Valhalla, Westchester County",Adzuna::API::Response::Location,NaN
2,2026-09-24T08:55:11Z,5896874781,0,Adzuna::API::Response::Job,120640.00,Data analyst,120640.00,"Data Analyst Location: Hybrid | Nashville, TN ...",-86.780596,eyJhbGciOiJIUzI1NiJ9.eyJzIjoiTmlidjlNLTY4UkdDd...,...,https://www.adzuna.com/land/ad/5896874781?se=N...,Adzuna::API::Response::Company,Vaco LLC,Adzuna::API::Response::Category,it-jobs,IT Jobs,"[US, Tennessee, Davidson County, Nashville]","Nashville, Davidson County",Adzuna::API::Response::Location,NaN
3,2026-09-16T16:12:22Z,5886631734,0,Adzuna::API::Response::Job,82400.00,Human Resources Data Analyst,110000.00,Arnold & Porter is an international law firm w...,-77.019023,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTg4NjYzMTczNCIsI...,...,https://www.adzuna.com/land/ad/5886631734?se=N...,Adzuna::API::Response::Company,Arnold Porter,Adzuna::API::Response::Category,hr-jobs,HR & Recruitment Jobs,"[US, Washington, D.C., Washington]","Washington, Washington, D.C.",Adzuna::API::Response::Location,full_time
4,2026-09-17T22:33:05Z,5888579132,1,Adzuna::API::Response::Job,78684.16,Data Analyst,78684.16,hackajob is collaborating with MANTECH to conn...,-81.311499,eyJhbGciOiJIUzI1NiJ9.eyJzIjoiTmlidjlNLTY4UkdDd...,...,https://www.adzuna.com/land/ad/5888579132?se=N...,Adzuna::API::Response::Company,MANTECH,Adzuna::API::Response::Category,it-jobs,IT Jobs,"[US, Georgia, Effingham County, Springfield]","Springfield, Effingham County",Adzuna::API::Response::Location,NaN


### Initial Observation

The API response contains both simple fields and nested attributes. Company and location information are stored as nested objects, while job title, description, posting date, and salary information are provided at the job-record level.

The next step will be to identify which fields are relevant for the analysis and evaluate their data quality.

## 7. Data Understanding

Before cleaning or transforming the API data, inspect the available fields, data types, sample records, and missing values to determine which attributes are relevant for the analysis.

### 7.1 Initial Data Quality Check

Before selecting or transforming fields, inspect missing values and data types across the raw API response to identify potential data quality issues.

In [40]:
df.isna().sum().sort_values(ascending=False)


contract_time            9
redirect_url             0
location.__CLASS__       0
location.display_name    0
location.area            0
category.label           0
category.tag             0
category.__CLASS__       0
company.display_name     0
company.__CLASS__        0
created                  0
id                       0
adref                    0
longitude                0
description              0
salary_max               0
title                    0
salary_min               0
__CLASS__                0
salary_is_predicted      0
latitude                 0
dtype: int64

In [42]:
(df.isna().mean() * 100).sort_values(ascending=False)


contract_time            90.0
redirect_url              0.0
location.__CLASS__        0.0
location.display_name     0.0
location.area             0.0
category.label            0.0
category.tag              0.0
category.__CLASS__        0.0
company.display_name      0.0
company.__CLASS__         0.0
created                   0.0
id                        0.0
adref                     0.0
longitude                 0.0
description               0.0
salary_max                0.0
title                     0.0
salary_min                0.0
__CLASS__                 0.0
salary_is_predicted       0.0
latitude                  0.0
dtype: float64

### 7.2 Data Types

Inspect the current data types to identify fields that may require transformation before analysis.

In [43]:
df.dtypes


created                   object
id                        object
salary_is_predicted       object
__CLASS__                 object
salary_min               float64
title                     object
salary_max               float64
description               object
longitude                float64
adref                     object
latitude                 float64
redirect_url              object
company.__CLASS__         object
company.display_name      object
category.__CLASS__        object
category.tag              object
category.label            object
location.area             object
location.display_name     object
location.__CLASS__        object
contract_time             object
dtype: object

### 7.3 Schema Summary

Create a compact summary of each field's data type and missing-value count before deciding which columns are relevant for the analysis.

In [44]:
schema_summary = pd.DataFrame(
    {
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "missing_count": df.isna().sum().values,
        "missing_pct": (df.isna().mean().values * 100).round(1),
    }
)

schema_summary


,column,dtype,missing_count,missing_pct
0,created,object,0,0.0
1,id,object,0,0.0
2,salary_is_predicted,object,0,0.0
3,__CLASS__,object,0,0.0
4,salary_min,float64,0,0.0
5,title,object,0,0.0
6,salary_max,float64,0,0.0
7,description,object,0,0.0
8,longitude,float64,0,0.0
9,adref,object,0,0.0


### Initial Observations

The exploratory sample will be used to identify potential schema and data-quality issues. Because the current request contains only 10 job postings, these observations are not treated as representative findings about the broader U.S. job market.

### 7.4 Inspect Key Field Values

Inspect the actual values stored in important analytical fields before deciding how they should be cleaned or transformed.

In [45]:
df["salary_is_predicted"].value_counts(dropna=False)


salary_is_predicted
1    7
0    3
Name: count, dtype: int64

In [46]:
df["contract_time"].value_counts(dropna=False)


contract_time
NaN          9
full_time    1
Name: count, dtype: int64

In [47]:
type(df.loc[0, "location.area"])


list

In [48]:
df.loc[0, "location.area"]


['US', 'Tennessee', 'Williamson County', 'Brentwood']

In [49]:
df["created"].head()


0    2026-09-25T21:59:05Z
1    2026-08-24T02:59:05Z
2    2026-09-24T08:55:11Z
3    2026-09-16T16:12:22Z
4    2026-09-17T22:33:05Z
Name: created, dtype: object

In [50]:
df[["salary_min", "salary_max"]].describe()


,salary_min,salary_max
count,10.000000,10.000000
mean,92829.507000,97589.507000
std,27349.213963,25337.961142
min,60000.000000,65915.280000
25%,77386.105000,79013.120000
50%,82275.205000,93696.635000
75%,107134.255000,109441.180000
max,148544.220000,148544.220000


In [51]:
df[["title", "salary_min", "salary_max", "salary_is_predicted"]]


,title,salary_min,salary_max,salary_is_predicted
0,Data Analyst,60000.00,80000.00,0
1,Privacy Program Data Analyst (HIPAA),76953.42,76953.42,1
2,Data analyst,120640.00,120640.00,0
3,Human Resources Data Analyst,82400.00,110000.00,0
4,Data Analyst,78684.16,78684.16,1
5,Data Analyst,82150.41,82150.41,1
6,Data Analyst,105242.86,105242.86,1
7,Data Analyst (Financials data Analyst),65915.28,65915.28,1
8,Data Analyst,148544.22,148544.22,1
9,Data Analyst,107764.72,107764.72,1


In [52]:
df[
    [
        "title",
        "company.display_name",
        "location.display_name",
        "created",
        "salary_min",
        "salary_max",
        "salary_is_predicted",
        "contract_time",
    ]
]


,title,company.display_name,location.display_name,created,salary_min,salary_max,salary_is_predicted,contract_time
0,Data Analyst,"Altair Data Resources, Inc.","Brentwood, Williamson County",2026-09-25T21:59:05Z,60000.00,80000.00,0,NaN
1,Privacy Program Data Analyst (HIPAA),WMCHealth,"Valhalla, Westchester County",2026-08-24T02:59:05Z,76953.42,76953.42,1,NaN
2,Data analyst,Vaco LLC,"Nashville, Davidson County",2026-09-24T08:55:11Z,120640.00,120640.00,0,NaN
3,Human Resources Data Analyst,Arnold Porter,"Washington, Washington, D.C.",2026-09-16T16:12:22Z,82400.00,110000.00,0,full_time
4,Data Analyst,MANTECH,"Springfield, Effingham County",2026-09-17T22:33:05Z,78684.16,78684.16,1,NaN
5,Data Analyst,MANTECH,"Tampa, Hillsborough County",2026-08-01T06:42:49Z,82150.41,82150.41,1,NaN
6,Data Analyst,Bet365,"Denver, Colorado",2026-09-09T19:49:00Z,105242.86,105242.86,1,NaN
7,Data Analyst (Financials data Analyst),"eSolutionsFirst, LLC","Vienna, Fairfax County",2026-09-16T12:07:46Z,65915.28,65915.28,1,NaN
8,Data Analyst,Na Oiwi Kane,"Mount Rainier, Prince George's County",2026-07-23T21:24:28Z,148544.22,148544.22,1,NaN
9,Data Analyst,"Judge Group, Inc.","Bridgeport, Montgomery County",2026-06-24T22:42:09Z,107764.72,107764.72,1,NaN


### Initial Data Quality Observations

The exploratory sample shows that most core job attributes are populated, while `contract_time` is highly sparse. Several fields also require closer inspection before transformation, including posting dates, salary prediction flags, and nested location information.

Because the current dataset contains only 10 records, these observations are used for schema validation rather than broader job-market conclusions.

### 7.5 Field Selection

Select the fields required to answer the project's business questions while preserving the original API response separately.

The analytical dataset focuses on job identity, posting time, role, company, location, salary, job description, and selected job metadata.

In [54]:
analysis_columns = [
    "id",
    "created",
    "title",
    "description",
    "company.display_name",
    "location.area",
    "location.display_name",
    "latitude",
    "longitude",
    "salary_min",
    "salary_max",
    "salary_is_predicted",
]

analysis_df = df[analysis_columns].copy()


In [55]:
analysis_df.shape


(10, 12)

### 7.6 Cleaning Plan

Before transforming the analytical dataset, define the required cleaning steps for each field based on its current structure and intended analytical use.

Planned transformations include:

- Convert `created` from string to datetime
- Standardize column names for SQL and Tableau
- Parse `location.area` into geographic fields where possible
- Validate salary ranges and retain `salary_is_predicted` for salary-quality checks
- Check job IDs for duplicates
- Inspect text fields used for skill and entry-level analysis

In [56]:
analysis_df = analysis_df.rename(
    columns={
        "company.display_name": "company",
        "location.display_name": "location",
        "location.area": "location_area",
    }
)


In [57]:
analysis_df.columns.tolist()


['id',
 'created',
 'title',
 'description',
 'company',
 'location_area',
 'location',
 'latitude',
 'longitude',
 'salary_min',
 'salary_max',
 'salary_is_predicted']

### 7.7 Convert Posting Time

Convert the API posting timestamp from text into a timezone-aware datetime field so it can be used for date filtering, trend analysis, and future time-based features.

In [59]:
analysis_df["created"] = pd.to_datetime(
    analysis_df["created"], utc=True, errors="coerce"
)

analysis_df["created"].dtype

datetime64[ns, UTC]

In [60]:
analysis_df["created"].isna().sum()


np.int64(0)

In [61]:
analysis_df["created"].head()


0   2026-09-25 21:59:05+00:00
1   2026-08-24 02:59:05+00:00
2   2026-09-24 08:55:11+00:00
3   2026-09-16 16:12:22+00:00
4   2026-09-17 22:33:05+00:00
Name: created, dtype: datetime64[ns, UTC]

### 7.8 Duplicate Check

Use the job ID as the primary identifier to check whether the API sample contains duplicate job postings.

In [62]:
analysis_df["id"].duplicated().sum()


np.int64(0)

In [63]:
print("Rows:", len(analysis_df))
print("Unique job IDs:", analysis_df["id"].nunique())


Rows: 10
Unique job IDs: 10


### 7.9 Salary Range Validation

Validate salary fields before using them in compensation analysis. Check for invalid or inconsistent ranges while retaining the salary prediction flag for later quality filtering.

In [64]:
(analysis_df["salary_min"] > analysis_df["salary_max"]).sum()


np.int64(0)

In [ ]:
(analysis_df["salary_min"] <= 0).sum()


np.int64(0)

In [67]:
(analysis_df["salary_max"] <= 0).sum()


np.int64(0)

In [68]:
analysis_df["salary_midpoint"] = (
    analysis_df["salary_min"] + analysis_df["salary_max"]
) / 2


In [69]:
analysis_df[
    ["salary_min", "salary_max", "salary_midpoint", "salary_is_predicted"]
].head()


,salary_min,salary_max,salary_midpoint,salary_is_predicted
0,60000.00,80000.00,70000.00,0
1,76953.42,76953.42,76953.42,1
2,120640.00,120640.00,120640.00,0
3,82400.00,110000.00,96200.00,0
4,78684.16,78684.16,78684.16,1


### 7.10 Inspect Location Structure

In [72]:
analysis_df["location_area"].apply(len).value_counts().sort_index()


location_area
3    2
4    8
Name: count, dtype: int64

### Validation Summary

The exploratory sample contains no duplicate job IDs and no invalid salary ranges. All salary values are positive, and no records have a minimum salary greater than the maximum salary.

Location hierarchy is not fully consistent: 8 records contain four geographic levels, while 2 records contain three. Location fields therefore require additional inspection before parsing.

### 7.11 Inspect Location Hierarchies

Because `location_area` contains lists of different lengths, inspect the shorter location hierarchies before extracting state, county, and city fields.

In [73]:
analysis_df[analysis_df["location_area"].apply(len) == 3][["location", "location_area"]]


,location,location_area
3,"Washington, Washington, D.C.","[US, Washington, D.C., Washington]"
6,"Denver, Colorado","[US, Colorado, Denver]"


In [74]:
analysis_df[analysis_df["location_area"].apply(len) == 4][
    ["location", "location_area"]
].head()


,location,location_area
0,"Brentwood, Williamson County","[US, Tennessee, Williamson County, Brentwood]"
1,"Valhalla, Westchester County","[US, New York, Westchester County, Valhalla]"
2,"Nashville, Davidson County","[US, Tennessee, Davidson County, Nashville]"
4,"Springfield, Effingham County","[US, Georgia, Effingham County, Springfield]"
5,"Tampa, Hillsborough County","[US, Florida, Hillsborough County, Tampa]"


### 7.12 Parse Location Fields

Convert the nested location hierarchy into separate geographic fields for downstream SQL analysis and Tableau visualization.

The sample contains two structures:

- 3 levels: Country → State → City
- 4 levels: Country → State → County → City

In [75]:
analysis_df["country"] = analysis_df["location_area"].apply(
    lambda x: x[0] if len(x) > 0 else None
)


In [76]:
analysis_df["state"] = analysis_df["location_area"].apply(
    lambda x: x[1] if len(x) > 1 else None
)


In [77]:
analysis_df["city"] = analysis_df["location_area"].apply(
    lambda x: x[-1] if len(x) > 0 else None
)


In [78]:
analysis_df["county"] = analysis_df["location_area"].apply(
    lambda x: x[-2] if len(x) >= 4 else None
)


In [79]:
analysis_df[["location", "location_area", "country", "state", "county", "city"]]


,location,location_area,country,state,county,city
0,"Brentwood, Williamson County","[US, Tennessee, Williamson County, Brentwood]",US,Tennessee,Williamson County,Brentwood
1,"Valhalla, Westchester County","[US, New York, Westchester County, Valhalla]",US,New York,Westchester County,Valhalla
2,"Nashville, Davidson County","[US, Tennessee, Davidson County, Nashville]",US,Tennessee,Davidson County,Nashville
3,"Washington, Washington, D.C.","[US, Washington, D.C., Washington]",US,"Washington, D.C.",None,Washington
4,"Springfield, Effingham County","[US, Georgia, Effingham County, Springfield]",US,Georgia,Effingham County,Springfield
5,"Tampa, Hillsborough County","[US, Florida, Hillsborough County, Tampa]",US,Florida,Hillsborough County,Tampa
6,"Denver, Colorado","[US, Colorado, Denver]",US,Colorado,None,Denver
7,"Vienna, Fairfax County","[US, Virginia, Fairfax County, Vienna]",US,Virginia,Fairfax County,Vienna
8,"Mount Rainier, Prince George's County","[US, Maryland, Prince George's County, Mount R...",US,Maryland,Prince George's County,Mount Rainier
9,"Bridgeport, Montgomery County","[US, Pennsylvania, Montgomery County, Bridgeport]",US,Pennsylvania,Montgomery County,Bridgeport


### 7.13 Final Cleaning Validation

Validate the cleaned analytical sample after datetime conversion, duplicate checks, salary validation, and geographic parsing.

In [80]:
analysis_df.shape


(10, 17)

In [81]:
analysis_df.dtypes


id                                  object
created                datetime64[ns, UTC]
title                               object
description                         object
company                             object
location_area                       object
location                            object
latitude                           float64
longitude                          float64
salary_min                         float64
salary_max                         float64
salary_is_predicted                 object
salary_midpoint                    float64
country                             object
state                               object
city                                object
county                              object
dtype: object

In [82]:
analysis_df.isna().sum().sort_values(ascending=False)


county                 2
salary_min             0
city                   0
state                  0
country                0
salary_midpoint        0
salary_is_predicted    0
salary_max             0
id                     0
created                0
latitude               0
location               0
location_area          0
company                0
description            0
title                  0
longitude              0
dtype: int64

In [83]:
analysis_df["id"].duplicated().sum()


np.int64(0)

In [84]:
print(
    "Invalid salary ranges:",
    (analysis_df["salary_min"] > analysis_df["salary_max"]).sum(),
)

print("Non-positive salary_min:", (analysis_df["salary_min"] <= 0).sum())

print("Non-positive salary_max:", (analysis_df["salary_max"] <= 0).sum())


Invalid salary ranges: 0
Non-positive salary_min: 0
Non-positive salary_max: 0


### 7.14 Final Analytical Sample

Review the cleaned sample that will serve as the prototype schema for the full job-market ingestion pipeline.

In [85]:
final_preview_columns = [
    "id",
    "created",
    "title",
    "company",
    "state",
    "county",
    "city",
    "salary_min",
    "salary_max",
    "salary_midpoint",
    "salary_is_predicted",
]

analysis_df[final_preview_columns].head(10)


,id,created,title,company,state,county,city,salary_min,salary_max,salary_midpoint,salary_is_predicted
0,5899042199,2026-09-25 21:59:05+00:00,Data Analyst,"Altair Data Resources, Inc.",Tennessee,Williamson County,Brentwood,60000.00,80000.00,70000.00,0
1,5854221865,2026-08-24 02:59:05+00:00,Privacy Program Data Analyst (HIPAA),WMCHealth,New York,Westchester County,Valhalla,76953.42,76953.42,76953.42,1
2,5896874781,2026-09-24 08:55:11+00:00,Data analyst,Vaco LLC,Tennessee,Davidson County,Nashville,120640.00,120640.00,120640.00,0
3,5886631734,2026-09-16 16:12:22+00:00,Human Resources Data Analyst,Arnold Porter,"Washington, D.C.",None,Washington,82400.00,110000.00,96200.00,0
4,5888579132,2026-09-17 22:33:05+00:00,Data Analyst,MANTECH,Georgia,Effingham County,Springfield,78684.16,78684.16,78684.16,1
5,5823867997,2026-08-01 06:42:49+00:00,Data Analyst,MANTECH,Florida,Hillsborough County,Tampa,82150.41,82150.41,82150.41,1
6,5877658057,2026-09-09 19:49:00+00:00,Data Analyst,Bet365,Colorado,None,Denver,105242.86,105242.86,105242.86,1
7,5886391534,2026-09-16 12:07:46+00:00,Data Analyst (Financials data Analyst),"eSolutionsFirst, LLC",Virginia,Fairfax County,Vienna,65915.28,65915.28,65915.28,1
8,5813141729,2026-07-23 21:24:28+00:00,Data Analyst,Na Oiwi Kane,Maryland,Prince George's County,Mount Rainier,148544.22,148544.22,148544.22,1
9,5775687128,2026-06-24 22:42:09+00:00,Data Analyst,"Judge Group, Inc.",Pennsylvania,Montgomery County,Bridgeport,107764.72,107764.72,107764.72,1


## 8. Summary

This notebook established and validated the initial data ingestion workflow for the U.S. Analytics Job Market Intelligence project.

Key steps completed:

- Connected securely to the Adzuna Jobs API
- Retrieved a small sample of current U.S. Data Analyst postings
- Inspected the JSON response structure
- Flattened nested JSON records into a Pandas DataFrame
- Evaluated missing values and field data types
- Selected fields based on the project's business questions
- Converted posting timestamps into datetime format
- Validated job IDs and salary ranges
- Created a salary midpoint metric
- Parsed nested location hierarchies into geographic fields

The 10-record dataset is used only for schema exploration and pipeline validation. The next stage will scale the ingestion process across multiple analytics roles and a larger number of job postings before conducting business analysis.